# AISC DeepFake — Region Fusion Experiments

Bu notebook **göz + kaş + ağız** bölgesel model çıktılarını birleştirir.

## Kullanılan 3 ortak model ailesi
1. **Swin V2 Tiny**
2. **EfficientNet-B0**
3. **Swin V2 Tiny + Texture Fusion**

## Kritik bilimsel kural
Göz, kaş ve ağız tarafındaki frame sayıları aynı olmak zorunda değildir. Fusion sırasında yalnızca **aynı kaynak frame'i temsil eden üç bölgenin kesişimi** kullanılır.

Notebook satır sırasına göre eşleştirme **YAPMAZ**. `image_path`, `sample_id`, `source_frame` veya benzeri alanlardan ortak bir `fusion_key` üretir ve inner join uygular.

> Not: Her model ailesi kendi içinde fuse edilir. Swin ile EfficientNet aynı fusion girdisine karıştırılmaz.

# Fusion 1 — Equal Soft Voting


Üç bölgenin fake olasılıkları eşit ağırlıkla ortalanır:

`p_fusion = (p_eye + p_brow + p_mouth) / 3`

Ayrı bir fusion modeli eğitmez. En sade baseline'dır.

In [4]:
# ============================================================
# 1) COLAB + CONFIG — DÜZELTİLMİŞ
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import json
import re
import math
import warnings
import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)


# ============================================================
# OUTPUT
# ============================================================

OUTPUT_ROOT = Path(
    "/content/drive/MyDrive/"
    "AISC DeepFake Çalışmaları/Deney 1/"
    "Kader/Deney 1/Sonuçlar/Fusion_Experiments"
)

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# ANA SONUÇ KLASÖRLERİ
# ============================================================

SEARCH_ROOTS = {

    "eye": Path(
        "/content/drive/MyDrive/"
        "AISC DeepFake Çalışmaları/Deney 1/"
        "Kader/Deney 1/Sonuçlar"
    ),

    "brow": Path(
        "/content/drive/MyDrive/"
        "AISC DeepFake Çalışmaları/Deney 1/"
        "Nazlıcan/Deney 1/Sonuçlar"
    ),

    "mouth": Path(
        "/content/drive/MyDrive/"
        "AISC DeepFake Çalışmaları/Deney 1/"
        "Dilara/Deney 1/Sonuçlar"
    ),
}


# ============================================================
# MODEL AİLELERİ
# ============================================================

MODEL_FAMILIES = {

    # --------------------------------------------------------
    # 1) SWIN V2 TINY
    # --------------------------------------------------------
    "swinv2_tiny": {

        "eye_test": (
            SEARCH_ROOTS["eye"]
            / "20260807_1031_eye_swinv2_tiny_seed42"
            / "predictions"
            / "test_frame_predictions.csv"
        ),

        "brow_test": (
            SEARCH_ROOTS["brow"]
            / "Kas_SwinV2_Tiny_Detayli_Sonuc_pdf"
            / "predictions"
            / "test_frame_predictions.csv"
        ),

        "mouth_test": (
            SEARCH_ROOTS["mouth"]
            / "20260807_2235_mouth_swinv2_tiny_seed42"
            / "predictions"
            / "test_frame_predictions.csv"
        ),

        "eye_val": None,
        "brow_val": None,
        "mouth_val": None,
    },


    # --------------------------------------------------------
    # 2) EFFICIENTNET-B0
    # --------------------------------------------------------
    "efficientnet_b0": {

        "eye_test": (
            SEARCH_ROOTS["eye"]
            / "20260808_0803_eye_efficientnet_b0_seed42"
            / "predictions"
            / "test_predictions.csv"
        ),

        "brow_test": (
            SEARCH_ROOTS["brow"]
            / "20260808_1248_eyebrow_efficientnet_b0_seed42"
            / "predictions"
            / "test_predictions.csv"
        ),

        "mouth_test": (
            SEARCH_ROOTS["mouth"]
            / "20260808_1257_mouth_efficientnet_b0_seed42"
            / "predictions"
            / "test_predictions.csv"
        ),

        "eye_val": None,
        "brow_val": None,
        "mouth_val": None,
    },


    # --------------------------------------------------------
    # 3) SWIN V2 TINY + TEXTURE FUSION
    # --------------------------------------------------------
    "swinv2_texture": {

    "eye_test": (
        SEARCH_ROOTS["eye"]
        / "20260806_1748_eye_swinv2_texturefusion_seed42"
        / "full"
        / "predictions"
        / "test_predictions.csv"
    ),

    "brow_test": (
        SEARCH_ROOTS["brow"]
        / "Swin V2-Tiny + LBP + GLCM + Gabor + Wavelet Fusion"
        / "predictions"
        / "test_predictions_frame_level.csv"
    ),

    "mouth_test": (
        SEARCH_ROOTS["mouth"]
        / "SwinV2_TextureFusion_Mouth"
        / "20260807_1550_mouth_swinv2_texturefusion_seed42"
        / "full"
        / "predictions"
        / "test_predictions.csv"
    ),

    "eye_val": None,
    "brow_val": None,
    "mouth_val": None,
},
}


# ============================================================
# PATH QUALITY GATE
# ============================================================

print("\n" + "=" * 90)
print("MODEL DOSYASI KONTROLÜ")
print("=" * 90)

all_test_paths_ok = True

for family, cfg in MODEL_FAMILIES.items():

    print(f"\n[{family}]")

    for region in ("eye", "brow", "mouth"):

        key = f"{region}_test"
        path = cfg[key]

        if path is None:
            status = "❌ NONE"
            all_test_paths_ok = False

        elif Path(path).is_file():
            status = "✅ BULUNDU"

        else:
            status = "❌ BULUNAMADI"
            all_test_paths_ok = False

        print(
            f"{region.upper():5s} | "
            f"{status} | "
            f"{path}"
        )


# ============================================================
# ÖZET
# ============================================================

print("\n" + "=" * 90)

if all_test_paths_ok:

    print("✅ TÜM TEST PREDICTION DOSYALARI BULUNDU.")
    print("Fusion aşamasına geçilebilir.")

else:

    print("⚠️ EN AZ BİR TEST DOSYASI BULUNAMADI.")
    print(
        "Yukarıdaki ❌ olan yolu kontrol et. "
        "Fusion hücresini henüz çalıştırma."
    )

print("=" * 90)


# ============================================================
# CONFIG ÖZETİ
# ============================================================

print("\nCONFIG:\n")

print(
    json.dumps(
        {
            family: {
                k: str(v) if v is not None else None
                for k, v in cfg.items()
            }
            for family, cfg in MODEL_FAMILIES.items()
        },
        indent=2,
        ensure_ascii=False,
    )
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

MODEL DOSYASI KONTROLÜ

[swinv2_tiny]
EYE   | ✅ BULUNDU | /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deney 1/Kader/Deney 1/Sonuçlar/20260807_1031_eye_swinv2_tiny_seed42/predictions/test_frame_predictions.csv
BROW  | ✅ BULUNDU | /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deney 1/Nazlıcan/Deney 1/Sonuçlar/Kas_SwinV2_Tiny_Detayli_Sonuc_pdf/predictions/test_frame_predictions.csv
MOUTH | ✅ BULUNDU | /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deney 1/Dilara/Deney 1/Sonuçlar/20260807_2235_mouth_swinv2_tiny_seed42/predictions/test_frame_predictions.csv

[efficientnet_b0]
EYE   | ✅ BULUNDU | /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deney 1/Kader/Deney 1/Sonuçlar/20260808_0803_eye_efficientnet_b0_seed42/predictions/test_predictions.csv
BROW  | ✅ BULUNDU | /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deney 1/Nazlıcan/Deney 1/Sonuçlar/20260

In [10]:
# ============================================================
# 2) ORTAK YARDIMCI FONKSİYONLAR
# ============================================================
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score, confusion_matrix
)
from sklearn.metrics import roc_curve, precision_recall_curve
import matplotlib.pyplot as plt

LABEL_CANDIDATES = [
    "true_label",
    "label",
    "label_int",
    "target",
    "y_true",
    "ground_truth",
    "class_id",
    "true_class",
]

PROB_CANDIDATES = [
    # SwinV2 Tiny
    "fake_probability",

    # EfficientNet-B0
    "prob_fake",

    # SwinV2 + Texture
    "probability_fake",

    # genel olası isimler
    "probability",
    "prob",
    "y_score",
    "score",
    "fake_prob",
    "prediction_probability",
]

KEY_CANDIDATES = [
    "source_frame",
    "relative_frame_path",
    "frame_path",
    "original_frame",

    # Swin / Texture
    "image_path",

    # EfficientNet
    "path",

    "sample_id",
    "frame_stem",
]

def _first_existing(cols, candidates):
    lower_map = {str(c).lower(): c for c in cols}
    for c in candidates:
        if c.lower() in lower_map:
            return lower_map[c.lower()]
    return None

def normalize_label(v):
    if pd.isna(v):
        return np.nan
    if isinstance(v, (int, np.integer, float, np.floating)):
        return int(float(v) >= 0.5)
    s = str(v).strip().lower()
    if s in {"1","fake","deepfake","manipulated","sahte","f"}:
        return 1
    if s in {"0","real","original","genuine","gerçek","r"}:
        return 0
    try:
        return int(float(s) >= 0.5)
    except:
        raise ValueError(f"Etiket çözülemedi: {v!r}")

def canonical_frame_key(value):
    """
    Göz, kaş ve ağız ROI dosyalarının farklı isimlendirme
    biçimlerinden ortak upstream frame kimliğini çıkarır.

    Örnekler
    --------
    Eye:
        fake_test_00000__face_00.jpg

    Brow:
        fake_test_00000.jpg

    Mouth:
        fake_test_fake_test_00000_face00.png

    Üçü de:
        fake_test_00000

    olarak normalize edilir.
    """

    if pd.isna(value):
        return None

    s = str(value).replace("\\", "/").strip().lower()

    # Sadece dosya adı
    name = s.split("/")[-1]

    # Uzantıyı kaldır
    name = re.sub(
        r"\.(jpg|jpeg|png|bmp|webp|npy)$",
        "",
        name,
        flags=re.IGNORECASE,
    )

    # --------------------------------------------------------
    # ANA KURAL
    # --------------------------------------------------------
    # real/fake + train/val/validation/test + frame numarası
    #
    # Örnek:
    # fake_test_00021
    # real_validation_00142
    # fake_train_01234
    #
    # Mouth'taki:
    # fake_test_fake_test_00021_face00
    #
    # içinde de ikinci "fake_test_00021" kısmını yakalar.
    # --------------------------------------------------------

    matches = re.findall(
        r"(real|fake)_(train|test|val|validation)_(\d+)",
        name,
        flags=re.IGNORECASE,
    )

    if matches:

        # Mouth isminde aynı desen birden fazla kez bulunabilir.
        # En sondaki gerçek frame kimliğini kullanıyoruz.
        label, split, frame_number = matches[-1]

        split = split.lower()

        # val / validation tek standarda getir
        if split == "validation":
            split = "val"

        # Sayıyı sabit 5 basamakta tut
        frame_number = str(int(frame_number)).zfill(5)

        return f"{label.lower()}_{split}_{frame_number}"

    # --------------------------------------------------------
    # FALLBACK
    # --------------------------------------------------------
    # Beklenmeyen isim formatı gelirse hatayı sessizce gizlemek
    # yerine dosya adını normalize ederek döndür.
    # Sonraki quality gate bunu gösterecek.
    # --------------------------------------------------------

    name = re.sub(
        r"(__)?face[_-]?\d+$",
        "",
        name,
        flags=re.IGNORECASE,
    )

    name = re.sub(
        r"_face\d+$",
        "",
        name,
        flags=re.IGNORECASE,
    )

    return name

def load_prediction_csv(path, region_name):
    if path is None:
        raise FileNotFoundError(
            f"{region_name}: prediction CSV yolu bulunamadı. CONFIG hücresinde yolu elle gir."
        )
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"{region_name}: dosya yok -> {path}")

    df = pd.read_csv(path)
    label_col = _first_existing(df.columns, LABEL_CANDIDATES)
    prob_col = _first_existing(df.columns, PROB_CANDIDATES)
    key_col = _first_existing(df.columns, KEY_CANDIDATES)

    if label_col is None:
        raise ValueError(f"{region_name}: gerçek etiket kolonu bulunamadı. Kolonlar={list(df.columns)}")
    if prob_col is None:
        raise ValueError(f"{region_name}: fake probability kolonu bulunamadı. Kolonlar={list(df.columns)}")
    if key_col is None:
        raise ValueError(f"{region_name}: örnek/frame kimliği kolonu bulunamadı. Kolonlar={list(df.columns)}")

    out = pd.DataFrame({
        "fusion_key": df[key_col].map(canonical_frame_key),
        "label": df[label_col].map(normalize_label).astype(int),
        f"p_{region_name}": pd.to_numeric(df[prob_col], errors="coerce"),
        f"raw_key_{region_name}": df[key_col].astype(str),
    }).dropna(subset=[f"p_{region_name}"])

    agg = out.groupby("fusion_key", as_index=False).agg({
        "label": ["min","max"],
        f"p_{region_name}": "mean",
        f"raw_key_{region_name}": "first",
    })
    agg.columns = [
        "fusion_key", "label_min", "label_max",
        f"p_{region_name}", f"raw_key_{region_name}"
    ]
    bad = agg[agg.label_min != agg.label_max]
    if len(bad):
        raise ValueError(
            f"{region_name}: aynı fusion_key altında çelişkili etiket bulundu. İlk örnekler:\n{bad.head()}"
        )
    agg["label"] = agg["label_min"].astype(int)
    return agg.drop(columns=["label_min","label_max"])

def align_three(eye_path, brow_path, mouth_path):
    e = load_prediction_csv(eye_path, "eye")
    b = load_prediction_csv(brow_path, "brow")
    m = load_prediction_csv(mouth_path, "mouth")

    x = e.merge(b, on="fusion_key", how="inner", suffixes=("_eye", "_brow"))
    x = x.merge(m, on="fusion_key", how="inner")

    label_cols = [c for c in x.columns if c.startswith("label")]
    if len(label_cols) < 3:
        raise RuntimeError(f"Beklenen etiket kolonları oluşmadı: {label_cols}")

    labels = x[label_cols].astype(int)
    ok = labels.nunique(axis=1).eq(1)
    if not ok.all():
        raise ValueError(f"{(~ok).sum()} ortak frame'de göz/kaş/ağız etiketleri uyuşmuyor.")

    x["label"] = labels.iloc[:,0].astype(int)
    x = x.drop(columns=label_cols)

    if len(x) == 0:
        raise RuntimeError(
            "Göz-kaş-ağız ortak frame kesişimi 0. canonical_frame_key() "
            "fonksiyonunu gerçek dosya adlarınıza göre revize edin."
        )

    print(
        f"Eye unique={len(e)} | Brow unique={len(b)} | Mouth unique={len(m)} | ORTAK={len(x)}"
    )
    print(x["label"].value_counts().sort_index().rename({0:"REAL",1:"FAKE"}))
    return x.sort_values("fusion_key").reset_index(drop=True)

def find_best_threshold(y, p, metric="balanced_accuracy"):
    grid = np.linspace(0.05, 0.95, 181)
    rows = []
    for t in grid:
        yp = (np.asarray(p) >= t).astype(int)
        if metric == "f1":
            s = f1_score(y, yp, zero_division=0)
        else:
            s = balanced_accuracy_score(y, yp)
        rows.append((t, s))
    return max(rows, key=lambda z: z[1])[0]

def compute_metrics(y, p, threshold=0.5):
    y = np.asarray(y).astype(int)
    p = np.asarray(p).astype(float)
    yp = (p >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, yp, labels=[0,1]).ravel()
    return {
        "n": int(len(y)),
        "threshold": float(threshold),
        "accuracy": accuracy_score(y, yp),
        "balanced_accuracy": balanced_accuracy_score(y, yp),
        "precision": precision_score(y, yp, zero_division=0),
        "recall": recall_score(y, yp, zero_division=0),
        "specificity": tn / (tn + fp) if (tn + fp) else np.nan,
        "f1": f1_score(y, yp, zero_division=0),
        "roc_auc": roc_auc_score(y, p) if len(np.unique(y)) == 2 else np.nan,
        "pr_auc": average_precision_score(y, p) if len(np.unique(y)) == 2 else np.nan,
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }

def save_standard_figures(df, prob_col, title, outdir, threshold):
    outdir = Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)
    y = df["label"].to_numpy()
    p = df[prob_col].to_numpy()

    fpr, tpr, _ = roc_curve(y, p)
    fig = plt.figure(figsize=(7,6))
    plt.plot(fpr, tpr, label=f"AUC={roc_auc_score(y,p):.4f}")
    plt.plot([0,1],[0,1],"--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"{title} — ROC Curve")
    plt.legend()
    plt.tight_layout()
    plt.savefig(outdir/"roc_curve.png", dpi=600)
    plt.close(fig)

    prec, rec, _ = precision_recall_curve(y, p)
    fig = plt.figure(figsize=(7,6))
    plt.plot(rec, prec, label=f"AP={average_precision_score(y,p):.4f}")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(f"{title} — Precision–Recall Curve")
    plt.legend()
    plt.tight_layout()
    plt.savefig(outdir/"precision_recall_curve.png", dpi=600)
    plt.close(fig)

    yp = (p >= threshold).astype(int)
    cm = confusion_matrix(y, yp, labels=[0,1])
    fig = plt.figure(figsize=(6,5))
    plt.imshow(cm)
    plt.xticks([0,1], ["REAL","FAKE"])
    plt.yticks([0,1], ["REAL","FAKE"])
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(f"{title} — Confusion Matrix")
    for i in range(2):
        for j in range(2):
            plt.text(j, i, str(cm[i,j]), ha="center", va="center")
    plt.tight_layout()
    plt.savefig(outdir/"confusion_matrix.png", dpi=600)
    plt.close(fig)

def save_summary(rows, method_name):
    summary = pd.DataFrame(rows)
    out = OUTPUT_ROOT / method_name
    out.mkdir(parents=True, exist_ok=True)
    summary.to_csv(out/"all_model_families_summary.csv", index=False)
    display(summary)
    return summary

In [11]:
# ============================================================
# FRAME KEY NORMALIZATION TEST
# ============================================================

examples = [
    "fake_test_00000__face_00.jpg",
    "fake_test_00000.jpg",
    "fake_test_fake_test_00000_face00.png",

    "real_test_00125__face_00.jpg",
    "real_test_00125.jpg",
    "real_test_real_test_00125_face00.png",
]

for x in examples:
    print(
        f"{x:50s} -> {canonical_frame_key(x)}"
    )

fake_test_00000__face_00.jpg                       -> fake_test_00000
fake_test_00000.jpg                                -> fake_test_00000
fake_test_fake_test_00000_face00.png               -> fake_test_00000
real_test_00125__face_00.jpg                       -> real_test_00125
real_test_00125.jpg                                -> real_test_00125
real_test_real_test_00125_face00.png               -> real_test_00125


In [12]:
# ============================================================
# GERÇEK CSV'LERDE ORTAK FRAME KONTROLÜ
# ============================================================

for family, cfg in MODEL_FAMILIES.items():

    print("\n" + "=" * 80)
    print("MODEL:", family)
    print("=" * 80)

    try:

        eye_df = load_prediction_csv(
            cfg["eye_test"],
            "eye",
        )

        brow_df = load_prediction_csv(
            cfg["brow_test"],
            "brow",
        )

        mouth_df = load_prediction_csv(
            cfg["mouth_test"],
            "mouth",
        )

        eye_keys = set(
            eye_df["fusion_key"]
        )

        brow_keys = set(
            brow_df["fusion_key"]
        )

        mouth_keys = set(
            mouth_df["fusion_key"]
        )

        common = (
            eye_keys
            & brow_keys
            & mouth_keys
        )

        print(
            "Eye unique   :",
            len(eye_keys),
        )

        print(
            "Brow unique  :",
            len(brow_keys),
        )

        print(
            "Mouth unique :",
            len(mouth_keys),
        )

        print(
            "ORTAK FRAME  :",
            len(common),
        )

        print(
            "\nİlk 10 ortak key:"
        )

        print(
            sorted(common)[:10]
        )

    except Exception as ex:

        print(
            "❌ HATA:",
            ex,
        )


MODEL: swinv2_tiny
Eye unique   : 292
Brow unique  : 196
Mouth unique : 292
ORTAK FRAME  : 196

İlk 10 ortak key:
['fake_test_00000', 'fake_test_00001', 'fake_test_00002', 'fake_test_00003', 'fake_test_00005', 'fake_test_00007', 'fake_test_00009', 'fake_test_00011', 'fake_test_00013', 'fake_test_00014']

MODEL: efficientnet_b0
Eye unique   : 302
Brow unique  : 196
Mouth unique : 292
ORTAK FRAME  : 0

İlk 10 ortak key:
[]

MODEL: swinv2_texture
Eye unique   : 292
Brow unique  : 196
Mouth unique : 292
ORTAK FRAME  : 196

İlk 10 ortak key:
['fake_test_00000', 'fake_test_00001', 'fake_test_00002', 'fake_test_00003', 'fake_test_00005', 'fake_test_00007', 'fake_test_00009', 'fake_test_00011', 'fake_test_00013', 'fake_test_00014']


In [13]:
# ============================================================
# 3) FUSION METHOD 1 — EQUAL SOFT VOTING
# ============================================================
METHOD_NAME = "01_equal_soft_voting"
all_rows = []

for family, cfg in MODEL_FAMILIES.items():
    print("\n" + "="*90)
    print("MODEL FAMILY:", family)
    print("="*90)

    test = align_three(cfg["eye_test"], cfg["brow_test"], cfg["mouth_test"])
    test["fusion_probability"] = (
        test["p_eye"] + test["p_brow"] + test["p_mouth"]
    ) / 3.0

    try:
        val = align_three(cfg["eye_val"], cfg["brow_val"], cfg["mouth_val"])
        val["fusion_probability"] = (
            val["p_eye"] + val["p_brow"] + val["p_mouth"]
        ) / 3.0
        threshold = find_best_threshold(
            val["label"], val["fusion_probability"], metric="balanced_accuracy"
        )
        threshold_source = "validation_balanced_accuracy"
    except Exception as ex:
        warnings.warn(
            f"{family}: validation prediction yok/uyuşmadı. Threshold=0.5 kullanılacak. "
            f"TEST üzerinden threshold seçilmedi. Detay: {ex}"
        )
        threshold = 0.5
        threshold_source = "fixed_0.5"

    family_out = OUTPUT_ROOT / METHOD_NAME / family
    family_out.mkdir(parents=True, exist_ok=True)

    for name, col in [
        ("eye_only", "p_eye"),
        ("brow_only", "p_brow"),
        ("mouth_only", "p_mouth"),
        ("fusion", "fusion_probability"),
    ]:
        met = compute_metrics(test["label"], test[col], threshold=threshold)
        all_rows.append({
            "model_family": family,
            "method": METHOD_NAME,
            "evaluation": name,
            "threshold_source": threshold_source,
            **met,
        })

    test.to_csv(family_out/"aligned_test_predictions.csv", index=False)

    fusion_metrics = compute_metrics(test["label"], test["fusion_probability"], threshold)
    with open(family_out/"metrics.json","w",encoding="utf-8") as f:
        json.dump({
            "method": METHOD_NAME,
            "model_family": family,
            "threshold_source": threshold_source,
            **fusion_metrics,
        }, f, indent=2, ensure_ascii=False)

    save_standard_figures(
        test, "fusion_probability",
        f"{family} — Equal Soft Voting",
        family_out/"figures", threshold
    )

summary = save_summary(all_rows, METHOD_NAME)


MODEL FAMILY: swinv2_tiny
Eye unique=292 | Brow unique=196 | Mouth unique=292 | ORTAK=196


KeyError: 'label'

In [ ]:
# ============================================================
# 4) FINAL QUALITY GATES
# ============================================================
method_dir = OUTPUT_ROOT / METHOD_NAME
assert method_dir.exists(), method_dir

summary_file = method_dir / "all_model_families_summary.csv"
assert summary_file.is_file(), summary_file

final_summary = pd.read_csv(summary_file)
required_metrics = [
    "accuracy", "balanced_accuracy", "precision", "recall",
    "specificity", "f1", "roc_auc", "pr_auc"
]
missing = [c for c in required_metrics if c not in final_summary.columns]
assert not missing, f"Eksik metrik kolonları: {missing}"

numeric = final_summary[required_metrics].apply(pd.to_numeric, errors="coerce")
assert np.isfinite(numeric.to_numpy()).all(), "NaN/Inf metrik bulundu."

print("QUALITY GATES: PASSED")
print("Output:", method_dir)
display(final_summary)

## Çıktı yapısı

Notebook tamamlandığında Drive'da:

```text
Fusion_Experiments/
└── <fusion_yöntemi>/
    ├── all_model_families_summary.csv
    ├── swinv2_tiny/
    ├── efficientnet_b0/
    └── swinv2_texture/
```

oluşur.

Her yöntemde **aynı üç model ailesi** çalıştırılır. Böylece sunumda fusion yöntemleri adil biçimde karşılaştırılabilir.